---
title: Lesson 4. iCite
license: CC-BY-4.0
github: https://github.com/OSU-Libraries-Research-Services/data_visualization
subject: Tutorial
authors:
  - name: Sarah Anne Murphy
    orcid: 0000-0002-7787-6890
    affiliations:
      - The Ohio State University Libraries
date: 2025-05-06
output_stderr: remove
---

The __[Relative Citation Ratio](https://doi.org/10.1371/journal.pbio.1002541)__ (RCR) evaluates the influence of a scientific article by analyzing its co-citation network. An RCR of **1.0** represents the median citation rate for articles in a given field. **Values above 1.0** indicate that an article is cited more frequently than average, suggesting greater impact or influence.

To begin this lesson, we'll use the __[NIH Reporter](https://reporter.nih.gov/)__ to identify an active project at **The Ohio State University** with associated publications.  We will then retrieve the **RCR values** for thos publications using the __[iCite](https://icite.od.nih.gov/)__ API.

## Data skills | concepts
- Working with APIs

## Learning objectives
1. Locate API documentation and identify key components required to formulate an API request
2. Parse the API response and store extracted data.

This tutorial is designed to support multi-session __[workshops](https://library.osu.edu/events?tid=921)__ hosted by The Ohio State University Libraries Research Commons. It assumes you already have a basic understanding of Python, including how to iterate through lists and dictionaries to extract data using a for loop. To learn basic Python concepts visit the __[Python - Mastering the Basics](./python-basics.ipynb)__ tutorial.

# LESSON 4

The __[NIH Reporter](https://reporter.nih.gov/)__ is a database of NIH-funded research projects that provides information about grants, contracts, publications, patents, and other outcomes resulting from NIH funding. 

## Step 1: Locate an active NIH project
- Go to the __[NIH Reporter](https://reporter.nih.gov/)__.
- Under **Fiscal Year**, select **Active Projects**.
- Under **Organization**, search for and select **Ohio State University**.

![nih_reporter_1_search.png](images/nih_reporter_1_search.png "Screenshot showing the NIH Reporter Interface with Active Projects entered under Fiscal Year and Ohio State University entered under Organization")

---


## Step 2: Browse active projects
- As of **May 6, 2025** there were **687 active projects** at The Ohio State University.
- Browse the list and click on a project that interests you.

![nih_reporter_2_select_project.png](images/nih_reporter_2_select_project.png "Screenshot of the results page showing funded projects with the project Molecular biophysics predoctoral training at the Ohio State University circled")

---

## Step 3: Review project details
- Read the **project abstract** to understand its focus.
- To check for related publications:
    -  Click on **Publications** in the left-hand menu, or
    -  Scroll to the bottom of the project page. 

![nih_reporter_3_select_publications.png](images/nih_reporter_3_select_publications.png "Screenshot of the details page for the project Molecular biophysics predoctoral training at The Ohio State University with Publications circled in the left screen menu")

---

## Step 4: Export the publications
- If the publications are listed, use the **Export** option to download the list for further analysis.

![nih_reporter_4_export_publications.png](images/nih_reporter_4_export_publications.png "Screenshot of the publications page for the project with the location of the Export icon circled")

---


:::{admonition} Exercise 1: Read .csv
:class: sidebar note
:icon: False

Use **Pandas** to read the exported publication file into Python.

```{tip}
Move the file from downloads to your project folder
```
:::

:::{seealso} Solution
:class: dropdown

```{code-cell}
import pandas as pd
df=pd.read_csv('data/Publications_10875609_6May2025_171210.csv') #Change the path to your downloaded file

```
:::

Each article indexed in __[PubMed](https://pubmed.ncbi.nlm.nih.gov/)__ has a unique PMID (PubMed Identifier). The exported file includes a `PMID` column.

```{code-cell}
df.columns.tolist()
```

:::{admonition} Exercise 2: List PMIDs
:class: sidebar note
:icon: False

Create a list of PMIDs found in the `PMID` column. To prepare the data:
- Remove any missing (null) PMIDs
- Convert each PMID from a float to integer, then to string.

:::

:::{seealso} Solution
:class: dropdown

```{code-cell}
import pandas as pd
df=pd.read_csv('data/Publications_10875609_6May2025_171210.csv') 
pmid_list=df.PMID.dropna().tolist()
pmids=[]
for pmid in pmid_list:
  pmids.append(str(int(pmid)))
pmids

```
:::


:::{admonition} Exercise 3: iCite API
:class: sidebar note
:icon: False

__[Launch iCite](https://icite.od.nih.gov/analysis)__. Scroll to the bottom and and review:
- **Web Policies and Notices*
- **Bulk Data and APIs**

On the **Bulk Data and APIs** page, locate the **Python code example** for querying the iCite API.

:::

:::{seealso} Solution
:class: dropdown

```{code-block}
import requests


response = requests.get(
    "/".join([
        "https://icite.od.nih.gov/api",
        "pubs",
        "23456789",
    ]),
)
pub = response.json()
print(pub)
```

:::

:::{admonition} Exercise 4: Adjust code
:class: sidebar note
:icon: False

The Python code example provided on the *Bulk Data and APIs* webpage uses the following statement to assemble the url:
```{code-block}
    "/".join([
        "https://icite.od.nih.gov/api",
        "pubs",
        "23456789",
    ]),
```

Adjust the code to extract the following fields for each PMID.
- pmid
- doi
- journal
- relative citation ratio
- cited by
- references
- title
- year



:::

:::{seealso} Solution
:class: dropdown

```{code-block}
import pandas as pd
import requests

results=pd.DataFrame(columns=['pmid','doi','journal','rcr','cited_by','references','title','year'])

df=pd.read_csv('Publications_10875609_6May2025_171210.csv') 
pmid_list=df.PMID.dropna().tolist()
pmids=[]
for pmid in pmid_list:
  pmids.append(str(int(pmid)))

count=1
for each_pmid in pmids:
    print(f"Starting record {count}: {each_pmid}")
    response = requests.get(
        "/".join([
        "https://icite.od.nih.gov/api",
        "pubs",
        each_pmid,
        ]),
        )   
    pub = response.json()
    citing_pmids=pub['cited_by']
    cited_by=[]
    for each_citation in citing_pmids:
        cited_by.append(str(each_citation))
    reference_pmids=pub['references']
    references=[]
    for each_reference in references:
        references.append(str(each_reference))

    row = {
            'pmid': pub['pmid'],
            'doi': pub['doi'],
            'journal': pub['journal'],
            'rcr': pub['relative_citation_ratio'],
            'cited_by': ','.join(cited_by),
            'references': ','.join(references),
            'title': pub['title'],
            'year': pub['year']
            }
    each_row=pd.DataFrame(row, index=[0])
    results=pd.concat([each_row, results], axis=0, ignore_index=True)
    count += 1
```

:::
